# 📦 Philippine Address Cleaning Pipeline
### (Pandas + Regex + RapidFuzz)

This notebook processes large datasets (~292k rows) to:
- Normalize addresses
- Extract components using regex
- Map barangays using alias tables
- Validate against dim_location
- Apply fuzzy matching fallback


In [62]:
# Install dependencies (run once if needed)
# !pip install pandas rapidfuzz openpyxl xlsxwriter tqdm

import pandas as pd
import numpy as np
import re
import time
import logging
from pathlib import Path
from rapidfuzz import process, fuzz
from tqdm.notebook import tqdm

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('ph_pipeline')
log.info('Imports OK')

13:47:17  INFO      Imports OK


## 📂 Load Data

In [63]:
# -- File paths --
BASE_DIR = Path('..') / '..' / 'data'   # notebook is in update_address/notebooks

INPUT_FILE   = str(BASE_DIR / 'sample'  / 'sample_org_address.xlsx')
DIM_LOCATION = str(BASE_DIR / 'mapping' / 'dim_location_20260316_v2.csv')
ALIAS_FILE   = str(BASE_DIR / 'utils'   / 'ph_address_alias_extended_v4.csv')
OUT_VERIFIED = str(BASE_DIR / 'output'  / 'verified_addresses.xlsx')
OUT_INVALID  = str(BASE_DIR / 'output'  / 'invalid_addresses.xlsx')

# -- Batch input ---------------------------------------------------------------
input_paths = [
    Path('../../data/sample/Structured_Philippine_Addresses_Unmatched/') / f'Structured_Philippine_Addresses_Unmatched_part_{i:03d}.xlsx'
    for i in range(51, 52)   # Adjust range to cover your batches, e.g. range(1, 101)
]

# -- Fuzzy-match thresholds (0-100) -------------------------------------------
CITY_SCORE_HIGH  = 88
CITY_SCORE_LOW   = 65
PROV_SCORE_HIGH  = 88
PROV_SCORE_LOW   = 60
BRGY_SCORE_MIN   = 75

# -- API settings kept for optional future use (currently disabled) -----------
PHOTON_URL = 'https://photon.komoot.io/api'
PHOTON_UA  = 'ph-address-pipeline/1.0 (research)'
BATCH_DELAY = 1.1
MAX_RETRIES = 1

# -- Run controls --------------------------------------------------------------
USE_API        = False         # Force fuzzy-only mode for maximum speed
API_QUERY_MODE = 'strict'      # Placeholder when API is re-enabled
MAX_ROWS       = None          # None = all rows; set e.g. 100 for quick test runs

# -- Quick path check ----------------------------------------------------------
if input_paths:
    print(f'  Batch mode enabled: {len(input_paths)} input files')
    for p in input_paths:
        status = 'OK' if p.exists() else 'NOT FOUND'
        print(f'  {status}  {p}')
else:
    status = 'OK' if Path(INPUT_FILE).exists() else 'NOT FOUND'
    print(f'  {status}  {INPUT_FILE}')

for f in [DIM_LOCATION, ALIAS_FILE]:
    status = 'OK' if Path(f).exists() else 'NOT FOUND'
    print(f'  {status}  {f}')
print(f'  USE_API = {USE_API} (fuzzy-only)')

def read_csv_with_fallback(path: str) -> pd.DataFrame:
    strict_encodings = ['utf-8', 'utf-8-sig']
    for enc in strict_encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue

    # Keep mostly UTF-8 files readable even if they contain a few broken bytes.
    try:
        log.warning(f'Loaded {path} using utf-8 with replacement for invalid bytes')
        return pd.read_csv(path, encoding='utf-8', encoding_errors='replace')
    except Exception:
        pass

    for enc in ['cp1252', 'latin1']:
        try:
            log.warning(f'Loaded {path} using fallback encoding: {enc}')
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue

    raise ValueError(f'Unable to decode CSV file: {path}')

# -- Load data -----------------------------------------------------------------
if input_paths:
    existing_paths = [p for p in input_paths if p.exists()]
    missing_paths = [p for p in input_paths if not p.exists()]

    for mp in missing_paths:
        log.warning(f'Missing batch file: {mp}')

    if not existing_paths:
        raise FileNotFoundError('No batch files found in input_paths.')

    frames = []
    for p in existing_paths:
        tmp = pd.read_excel(p)
        tmp['_batch_file'] = p.name
        frames.append(tmp)
    df = pd.concat(frames, ignore_index=True)
else:
    df = pd.read_excel(INPUT_FILE)

if MAX_ROWS:
    df = df.iloc[:MAX_ROWS].copy()
    log.info(f'Limiting to {MAX_ROWS} rows (MAX_ROWS is set)')

dim_location = read_csv_with_fallback(DIM_LOCATION)
alias = read_csv_with_fallback(ALIAS_FILE)

log.info(f'Loaded input rows: {len(df):,}')
df.head()

13:47:17  WARNING   Loaded ..\..\data\mapping\dim_location_20260316_v2.csv using utf-8 with replacement for invalid bytes


  Batch mode enabled: 1 input files
  OK  ..\..\data\sample\Structured_Philippine_Addresses_Unmatched\Structured_Philippine_Addresses_Unmatched_part_051.xlsx
  OK  ..\..\data\mapping\dim_location_20260316_v2.csv
  OK  ..\..\data\utils\ph_address_alias_extended_v4.csv
  USE_API = False (fuzzy-only)


13:47:17  WARNING   Loaded ..\..\data\utils\ph_address_alias_extended_v4.csv using utf-8 with replacement for invalid bytes
13:47:17  INFO      Loaded input rows: 1,000


,Original_Address,_batch_file
0,BLDG C-19 RAWIS COMPOUND TONDO,Structured_Philippine_Addresses_Unmatched_part...
1,AIMATCO Builders,Structured_Philippine_Addresses_Unmatched_part...
2,Studio TV 5,Structured_Philippine_Addresses_Unmatched_part...
3,Pingkian 2 Tandang Sora,Structured_Philippine_Addresses_Unmatched_part...
4,happy homes centro brgy lag on dcn,Structured_Philippine_Addresses_Unmatched_part...


## 🧹 Normalize Text

In [64]:
def normalize_text(text):
    if pd.isna(text):
        return ''
    text = text.upper()
    text = re.sub(r'LANDMARK.*', '', text)
    text = re.sub(r'[^A-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_address'] = df['Original_Address'].apply(normalize_text)

## 🔍 Extract Components (Vectorized Regex)

In [65]:
# House number
df['House_Number'] = df['clean_address'].str.extract(r'^(\d+)')

# Street name
df['Street_Name'] = df['clean_address'].str.extract(r'\d+\s+([A-Z\s]+?)\s+(TIMOG|CAA|BARANGAY|BRGY|CITY)', expand=False)[0]

# Barangay candidates
df['Barangay_raw'] = df['clean_address'].str.extract(r'\b(TIMOG CAA|CAA|TIMOG)\b')

# City
df['City'] = df['clean_address'].str.extract(r'(LAS\s*PINAS\s*CITY)')

## 🔄 Normalize Barangay using Alias Table

In [66]:
# Support both alias file schemas:
# 1) alias / standard_name
# 2) pattern / replacement
if {'alias', 'standard_name'}.issubset(alias.columns):
    key_col, val_col = 'alias', 'standard_name'
elif {'pattern', 'replacement'}.issubset(alias.columns):
    key_col, val_col = 'pattern', 'replacement'
else:
    raise KeyError(
        "Alias file must contain either ['alias','standard_name'] or ['pattern','replacement'] columns. "
        f'Found: {list(alias.columns)}'
    )

alias_key = alias[key_col].fillna('').astype(str).str.upper().str.strip()
alias_val = alias[val_col].fillna('').astype(str).str.upper().str.strip()
alias_dict = {k: v for k, v in zip(alias_key, alias_val) if k}

def map_alias(val):
    if pd.isna(val):
        return None
    v = str(val).upper().strip()
    return alias_dict.get(v, v)

df['Barangay'] = df['Barangay_raw'].apply(map_alias)

## 🗺️ Exact Matching with dim_location

In [67]:
def canon_text(text: str) -> str:
    text = '' if pd.isna(text) else str(text).upper()
    text = re.sub(r'[^A-Z0-9\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def pick_col(df_obj: pd.DataFrame, candidates: list[str], required: bool = True):
    for c in candidates:
        if c in df_obj.columns:
            return c
    if required:
        raise KeyError(f'Missing expected columns. Tried: {candidates}. Found: {list(df_obj.columns)}')
    return None

# -- Detect dim_location schema ------------------------------------------------
DIM_BRGY_COL = pick_col(dim_location, ['barangay', 'barangay_name'])
DIM_CITY_COL = pick_col(dim_location, ['city', 'city_name'])
DIM_PROV_COL = pick_col(dim_location, ['province', 'province_name'])
DIM_REGION_COL = pick_col(dim_location, ['region_name', 'region'])

DIM_BRGY_CODE_COL = pick_col(dim_location, ['barangay_code'], required=False)
DIM_CITY_CODE_COL = pick_col(dim_location, ['city_code'], required=False)
DIM_PROV_CODE_COL = pick_col(dim_location, ['province_code'], required=False)
DIM_REGION_CODE_COL = pick_col(dim_location, ['region_code'], required=False)
DIM_PSGC_COL = pick_col(dim_location, ['psgc_10_digit', 'psgc_10'], required=False)

dim_work = dim_location.copy()
for c in [DIM_BRGY_COL, DIM_CITY_COL, DIM_PROV_COL, DIM_REGION_COL]:
    dim_work[c] = dim_work[c].fillna('').astype(str).str.upper().str.strip()
dim_work['_city_canon'] = dim_work[DIM_CITY_COL].apply(canon_text)
dim_work['_brgy_canon'] = dim_work[DIM_BRGY_COL].apply(canon_text)
dim_work['_prov_canon'] = dim_work[DIM_PROV_COL].apply(canon_text)

# -- Build alias replacer from table ------------------------------------------
alias_pairs = []
for k, v in alias_dict.items():
    kk = canon_text(k)
    vv = canon_text(v)
    if kk and vv and kk != vv:
        alias_pairs.append((kk, vv))
alias_pairs = sorted(set(alias_pairs), key=lambda x: -len(x[0]))

if alias_pairs:
    alias_pattern = re.compile(r'\b(' + '|'.join(re.escape(k) for k, _ in alias_pairs) + r')\b')
    alias_map = dict(alias_pairs)
else:
    alias_pattern = None
    alias_map = {}

def apply_alias(text: str) -> str:
    base = canon_text(text)
    if not alias_pattern:
        return base
    return alias_pattern.sub(lambda m: alias_map.get(m.group(0), m.group(0)), base)

# -- Canonical lookup sets -----------------------------------------------------
city_values = sorted(x for x in dim_work['_city_canon'].unique().tolist() if x)
province_values = sorted(x for x in dim_work['_prov_canon'].unique().tolist() if x)
barangay_values = sorted(x for x in dim_work['_brgy_canon'].unique().tolist() if x)

# Build city variants so CITY OF X / X CITY / X are interpreted consistently.
city_variant_map = {}
for city in city_values:
    variants = {city}
    if city.startswith('CITY OF '):
        core = city[len('CITY OF '):].strip()
        if core:
            variants.add(core)
            variants.add(f'{core} CITY')
    if city.endswith(' CITY'):
        core = city[:-len(' CITY')].strip()
        if core:
            variants.add(core)
            variants.add(f'CITY OF {core}')
    if city.startswith('MUNICIPALITY OF '):
        core = city[len('MUNICIPALITY OF '):].strip()
        if core:
            variants.add(core)
    for v in variants:
        if v and v not in city_variant_map:
            city_variant_map[v] = city

city_variants_sorted = sorted(city_variant_map.keys(), key=len, reverse=True)
province_values_sorted = sorted(province_values, key=len, reverse=True)

# Build explicit barangay matcher using n-gram exact matches.
brgy_token_max = max((len(x.split()) for x in barangay_values), default=1)
barangay_set = set(barangay_values)
barangay_to_cityset = {}
for _, r in dim_work[['_brgy_canon', '_city_canon']].drop_duplicates().iterrows():
    b = r['_brgy_canon']
    c = r['_city_canon']
    barangay_to_cityset.setdefault(b, set()).add(c)

def explicit_ngrams(text: str, max_n: int):
    toks = text.split()
    out = set()
    for n in range(1, max_n + 1):
        for i in range(0, max(0, len(toks) - n + 1)):
            out.add(' '.join(toks[i:i+n]))
    return out

def find_explicit_city(address_norm: str):
    padded = f' {address_norm} '
    for v in city_variants_sorted:
        if f' {v} ' in padded:
            return city_variant_map[v], v, 'city_text'
    return None, None, None

def find_explicit_province(address_norm: str):
    padded = f' {address_norm} '
    for p in province_values_sorted:
        if f' {p} ' in padded:
            return p
    return None

# Curated explicit district/area cues (strict exact phrase matches only).
AREA_TO_CITY_RAW = {
    # Manila districts
    'TONDO': ['MANILA', 'CITY OF MANILA'],
    'BINONDO': ['MANILA', 'CITY OF MANILA'],
    'SAN NICOLAS': ['MANILA', 'CITY OF MANILA'],
    'SANTA CRUZ': ['MANILA', 'CITY OF MANILA'],
    'QUIAPO': ['MANILA', 'CITY OF MANILA'],
    'SAMPALOC': ['MANILA', 'CITY OF MANILA'],
    'SAN MIGUEL': ['MANILA', 'CITY OF MANILA'],
    'ERMITA': ['MANILA', 'CITY OF MANILA'],
    'INTRAMUROS': ['MANILA', 'CITY OF MANILA'],
    'MALATE': ['MANILA', 'CITY OF MANILA'],
    'PACO': ['MANILA', 'CITY OF MANILA'],
    'PANDACAN': ['MANILA', 'CITY OF MANILA'],
    'PORT AREA': ['MANILA', 'CITY OF MANILA'],
    'SANTA ANA': ['MANILA', 'CITY OF MANILA'],
    # Quezon City
    'CUBAO': ['QUEZON CITY', 'CITY OF QUEZON'],
    'COMMONWEALTH': ['QUEZON CITY', 'CITY OF QUEZON'],
    'DILIMAN': ['QUEZON CITY', 'CITY OF QUEZON'],
    'NEW MANILA': ['QUEZON CITY', 'CITY OF QUEZON'],
    'FAIRVIEW': ['QUEZON CITY', 'CITY OF QUEZON'],
    'NOVALICHES': ['QUEZON CITY', 'CITY OF QUEZON'],
    'PROJECT 4': ['QUEZON CITY', 'CITY OF QUEZON'],
    # Taguig
    'BGC': ['CITY OF TAGUIG', 'TAGUIG'],
    'BONIFACIO GLOBAL CITY': ['CITY OF TAGUIG', 'TAGUIG'],
    # Parañaque / Las Piñas
    'SUN VALLEY': ['CITY OF PARAANAQUE', 'CITY OF PARANAQUE', 'PARANAQUE', 'PARANAQUE CITY'],
    'ALMANZA': ['CITY OF LAS PINAS', 'LAS PINAS CITY', 'LAS PINAS'],
    # Makati / Mandaluyong / San Juan (clear area cues)
    'LEGASPI VILLAGE': ['CITY OF MAKATI', 'MAKATI CITY', 'MAKATI'],
    'SALCEDO VILLAGE': ['CITY OF MAKATI', 'MAKATI CITY', 'MAKATI'],
    'POBLACION MAKATI': ['CITY OF MAKATI', 'MAKATI CITY', 'MAKATI'],
    'GREENHILLS': ['CITY OF SAN JUAN', 'SAN JUAN CITY', 'SAN JUAN'],
    'WACK WACK': ['CITY OF MANDALUYONG', 'MANDALUYONG CITY', 'MANDALUYONG'],
}

def resolve_city_candidate(candidates: list[str]):
    for cand in candidates:
        cc = canon_text(cand)
        if cc in city_values:
            return cc
        if cc in city_variant_map:
            return city_variant_map[cc]
    return None

AREA_TO_CITY = {}
for area, city_cands in AREA_TO_CITY_RAW.items():
    city_resolved = resolve_city_candidate(city_cands)
    if city_resolved:
        AREA_TO_CITY[canon_text(area)] = city_resolved

area_terms_sorted = sorted(AREA_TO_CITY.keys(), key=len, reverse=True)

def find_explicit_area_city(address_norm: str):
    padded = f' {address_norm} '
    for area in area_terms_sorted:
        if f' {area} ' in padded:
            return AREA_TO_CITY[area], area, 'area_text'
    return None, None, None

def find_explicit_barangays(address_norm: str):
    grams = explicit_ngrams(address_norm, brgy_token_max)
    return sorted([g for g in grams if g in barangay_set], key=len, reverse=True)

def find_barangay(address_norm: str, subset: pd.DataFrame, brgy_hint: str | None):
    brgys = sorted(x for x in subset['_brgy_canon'].unique().tolist() if x)
    if not brgys:
        return None, 0, None

    padded = f' {address_norm} '
    for b in sorted(brgys, key=len, reverse=True):
        if f' {b} ' in padded:
            return b, 100, 'exact'

    search_text = canon_text(brgy_hint) if brgy_hint else address_norm
    m = process.extractOne(search_text, brgys, scorer=fuzz.WRatio, score_cutoff=BRGY_SCORE_MIN)
    if m:
        return m[0], int(m[1]), 'fuzzy'
    return None, 0, None

# -- Strict hierarchical matching ----------------------------------------------
records = []
for _, row in tqdm(df.iterrows(), total=len(df), desc='Hierarchical match', unit='row'):
    rec = row.to_dict()
    rec.setdefault('Original_Address', rec.get('original_address', rec.get('Original_Address')))

    raw_text = rec.get('clean_address', rec.get('Original_Address', ''))
    addr_norm = apply_alias(raw_text)
    rec['address_normalized'] = addr_norm

    explicit_city, city_evidence, city_source = find_explicit_city(addr_norm)
    explicit_province = find_explicit_province(addr_norm)
    area_city, area_evidence, area_source = find_explicit_area_city(addr_norm)
    explicit_brgy_hits = find_explicit_barangays(addr_norm)

    brgy_hint = rec.get('Barangay', None)
    has_brgy_hint = bool(
        pd.notna(brgy_hint) and str(brgy_hint).strip()
    ) or bool(re.search(r'\b(BRGY|BARANGAY|BGY)\b', addr_norm))

    # Rule A: if no explicit city but explicit unambiguous barangay exists,
    # resolve via dim_location without assumption keywords.
    if not explicit_city and explicit_brgy_hits:
        unique_city_from_brgy = None
        matched_brgy = None
        for b in explicit_brgy_hits:
            city_set = barangay_to_cityset.get(b, set())
            if len(city_set) == 1:
                unique_city_from_brgy = next(iter(city_set))
                matched_brgy = b
                break

        if unique_city_from_brgy:
            explicit_city = unique_city_from_brgy
            city_evidence = matched_brgy
            city_source = 'barangay_text'
        elif area_city:
            explicit_city = area_city
            city_evidence = area_evidence
            city_source = area_source
        else:
            rec.update({
                'barangay': None, 'barangay_code': None,
                'city': None, 'city_code': None,
                'province': explicit_province, 'province_code': None,
                'region_name': None, 'region_code': None,
                'psgc_10_digit': None,
                'city_score': 0,
                'province_score': 100 if explicit_province else 0,
                'confidence': 'low',
                'is_valid': False,
                'invalid_reason': 'barangay_ambiguous_no_city',
                'city_signal': None,
            })
            records.append(rec)
            continue

    # Rule B: if no explicit city yet, allow strict curated area evidence.
    if not explicit_city and area_city:
        explicit_city = area_city
        city_evidence = area_evidence
        city_source = area_source

    # Rule C (requested): no explicit city and no resolvable explicit barangay => invalid.
    if not explicit_city:
        rec.update({
            'barangay': None, 'barangay_code': None,
            'city': None, 'city_code': None,
            'province': explicit_province, 'province_code': None,
            'region_name': None, 'region_code': None,
            'psgc_10_digit': None,
            'city_score': 0,
            'province_score': 100 if explicit_province else 0,
            'confidence': 'low',
            'is_valid': False,
            'invalid_reason': 'missing_explicit_city',
            'city_signal': None,
        })
        records.append(rec)
        continue

    city_subset = dim_work[dim_work['_city_canon'] == explicit_city].copy()
    if city_subset.empty:
        rec.update({
            'barangay': None, 'barangay_code': None,
            'city': explicit_city, 'city_code': None,
            'province': explicit_province, 'province_code': None,
            'region_name': None, 'region_code': None,
            'psgc_10_digit': None,
            'city_score': 90,
            'province_score': 100 if explicit_province else 0,
            'confidence': 'low',
            'is_valid': False,
            'invalid_reason': 'city_not_in_dim',
            'city_signal': explicit_city,
        })
        records.append(rec)
        continue

    # Tightened province consistency when province is explicitly mentioned.
    if explicit_province:
        city_provs = set(city_subset['_prov_canon'].dropna().unique().tolist())
        if explicit_province not in city_provs:
            rec.update({
                'barangay': None, 'barangay_code': None,
                'city': explicit_city, 'city_code': None,
                'province': explicit_province, 'province_code': None,
                'region_name': None, 'region_code': None,
                'psgc_10_digit': None,
                'city_score': 95,
                'province_score': 100,
                'confidence': 'low',
                'is_valid': False,
                'invalid_reason': 'province_city_conflict',
                'city_signal': explicit_city,
            })
            records.append(rec)
            continue

        city_subset = city_subset[city_subset['_prov_canon'] == explicit_province].copy()
        if city_subset.empty:
            rec.update({
                'barangay': None, 'barangay_code': None,
                'city': explicit_city, 'city_code': None,
                'province': explicit_province, 'province_code': None,
                'region_name': None, 'region_code': None,
                'psgc_10_digit': None,
                'city_score': 95,
                'province_score': 100,
                'confidence': 'low',
                'is_valid': False,
                'invalid_reason': 'province_city_conflict',
                'city_signal': explicit_city,
            })
            records.append(rec)
            continue

    brgy_match, brgy_score, brgy_method = find_barangay(addr_norm, city_subset, brgy_hint)

    chosen = city_subset.iloc[0]
    if brgy_match:
        brgy_rows = city_subset[city_subset['_brgy_canon'] == brgy_match]
        if not brgy_rows.empty:
            chosen = brgy_rows.iloc[0]

    rec.update({
        'barangay': chosen[DIM_BRGY_COL] if brgy_match else None,
        'barangay_code': (chosen[DIM_BRGY_CODE_COL] if DIM_BRGY_CODE_COL else None) if brgy_match else None,
        'city': chosen[DIM_CITY_COL],
        'city_code': chosen[DIM_CITY_CODE_COL] if DIM_CITY_CODE_COL else None,
        'province': chosen[DIM_PROV_COL],
        'province_code': chosen[DIM_PROV_CODE_COL] if DIM_PROV_CODE_COL else None,
        'region_name': chosen[DIM_REGION_COL],
        'region_code': chosen[DIM_REGION_CODE_COL] if DIM_REGION_CODE_COL else None,
        'psgc_10_digit': chosen[DIM_PSGC_COL] if DIM_PSGC_COL else None,
        'city_score': 100 if city_source in {'city_text', 'barangay_text'} else 95,
        'province_score': 100 if explicit_province else 90,
        'city_signal': explicit_city,
    })

    # Requested: if city is explicit and no barangay mentioned, keep valid.
    if has_brgy_hint and not brgy_match:
        rec['confidence'] = 'low'
        rec['is_valid'] = False
        rec['invalid_reason'] = 'barangay_unresolved'
    else:
        rec['confidence'] = 'high'
        rec['is_valid'] = True
        rec['invalid_reason'] = None

    rec['barangay_score'] = brgy_score
    records.append(rec)

merged = pd.DataFrame(records)
log.info(f'Matched rows: {len(merged):,}')
print('Validity distribution:')
print(merged['is_valid'].value_counts(dropna=False).to_string())
if 'invalid_reason' in merged.columns:
    print('\nInvalid reason distribution:')
    print(merged['invalid_reason'].fillna('resolved').value_counts().to_string())

Hierarchical match:   0%|          | 0/1000 [00:00<?, ?row/s]

13:47:23  INFO      Matched rows: 1,000


Validity distribution:
is_valid
False    609
True     391

Invalid reason distribution:
invalid_reason
missing_explicit_city         440
resolved                      391
barangay_ambiguous_no_city    165
barangay_unresolved             4


## ⚡ Fuzzy Matching (Fallback)

In [68]:
# Fuzzy recovery pass for rows that have a city but unresolved barangay
if 'invalid_reason' not in merged.columns:
    merged['invalid_reason'] = None

recover_mask = (
    merged['is_valid'].eq(False)
    & merged['invalid_reason'].eq('barangay_unresolved')
    & merged['city'].notna()
    & merged['city'].astype(str).str.strip().ne('')
)

recovered = 0
for idx in merged[recover_mask].index:
    city_val = canon_text(merged.at[idx, 'city'])
    subset = dim_work[dim_work['_city_canon'] == city_val]
    if subset.empty:
        continue

    brgys = sorted(x for x in subset['_brgy_canon'].unique().tolist() if x)
    if not brgys:
        continue

    addr_text = merged.at[idx, 'address_normalized']
    m = process.extractOne(addr_text, brgys, scorer=fuzz.WRatio, score_cutoff=max(BRGY_SCORE_MIN, 85))
    if not m:
        continue

    bcanon, bscore = m[0], int(m[1])
    hit = subset[subset['_brgy_canon'] == bcanon]
    if hit.empty:
        continue

    row = hit.iloc[0]
    merged.at[idx, 'barangay'] = row[DIM_BRGY_COL]
    merged.at[idx, 'barangay_code'] = row[DIM_BRGY_CODE_COL] if DIM_BRGY_CODE_COL else None
    merged.at[idx, 'psgc_10_digit'] = row[DIM_PSGC_COL] if DIM_PSGC_COL else None
    merged.at[idx, 'barangay_score'] = bscore
    merged.at[idx, 'confidence'] = 'medium'
    merged.at[idx, 'is_valid'] = True
    merged.at[idx, 'invalid_reason'] = None
    recovered += 1

print(f'Recovered rows by barangay fuzzy fallback: {recovered:,}')
print('Updated validity distribution:')
print(merged['is_valid'].value_counts(dropna=False).to_string())

Recovered rows by barangay fuzzy fallback: 2
Updated validity distribution:
is_valid
False    607
True     393


## 💾 Save Output

In [69]:
# -- Output targets ------------------------------------------------------------
out_root = Path(BASE_DIR) / 'output'
out_verified_dir = out_root / 'verified'
out_invalid_dir = out_root / 'invalid'
out_verified_dir.mkdir(parents=True, exist_ok=True)
out_invalid_dir.mkdir(parents=True, exist_ok=True)

# Build suffix from detected input parts (e.g. parts_001_010)
batch_nums = []
for p in input_paths:
    m = re.search(r'part_(\d+)', p.name.lower())
    if m:
        batch_nums.append(int(m.group(1)))

if batch_nums:
    batch_suffix = f'parts_{min(batch_nums):03d}_{max(batch_nums):03d}'
elif input_paths:
    batch_suffix = f'batch_{len(input_paths):03d}'
else:
    batch_suffix = 'single'

VERIFIED_FILE = str(out_verified_dir / f'verified_addresses_{batch_suffix}.xlsx')
INVALID_FILE = str(out_invalid_dir / f'invalid_addresses_{batch_suffix}.xlsx')
COMBINED_FILE = str(out_root / f'combined_addresses_{batch_suffix}.xlsx')

# -- Prepare base output -------------------------------------------------------
out_df = merged.copy()
if 'Original_Address' not in out_df.columns and 'original_address' in out_df.columns:
    out_df = out_df.rename(columns={'original_address': 'Original_Address'})
elif 'Original_Address' not in out_df.columns and 'clean_address' in out_df.columns:
    out_df['Original_Address'] = out_df['clean_address']

if 'address_normalized' not in out_df.columns and 'clean_address' in out_df.columns:
    out_df['address_normalized'] = out_df['clean_address']

# Validity fallback for this regex notebook profile.
if 'is_valid' not in out_df.columns:
    signal_cols = [c for c in ['barangay', 'city', 'province', 'region_name'] if c in out_df.columns]
    if signal_cols:
        out_df['is_valid'] = out_df[signal_cols].notna().any(axis=1)
    else:
        out_df['is_valid'] = False

BASE_EXPORT_COLS = [
    'Original_Address',
    'barangay_code', 'barangay',
    'city_code', 'city',
    'province_code', 'province',
    'region_code', 'region_name',
]

for c in BASE_EXPORT_COLS:
    if c not in out_df.columns:
        out_df[c] = None

verified_df = out_df[out_df['is_valid']][BASE_EXPORT_COLS].reset_index(drop=True)
invalid_df = out_df[~out_df['is_valid']][BASE_EXPORT_COLS].reset_index(drop=True)

# Combined file keeps additional diagnostics + status
combined_cols = [
    'Original_Address',
    'Normalized_Address',
    'barangay_code', 'barangay',
    'city_code', 'city',
    'province_code', 'province',
    'region_code', 'region_name',
    'psgc_10',
    'confidence',
    'city_score',
    'province_score',
    'osm_display',
    'osm_lat',
    'osm_lon',
    'status',
]

combined_df = out_df.copy()
combined_df['Normalized_Address'] = combined_df.get('address_normalized')
combined_df['psgc_10'] = combined_df.get('psgc_10_digit')
combined_df['status'] = np.where(combined_df['is_valid'], 'verified', 'invalid')
for c in combined_cols:
    if c not in combined_df.columns:
        combined_df[c] = None
combined_df = combined_df[combined_cols].reset_index(drop=True)


def write_xlsx(df: pd.DataFrame, path: str, sheet_name: str,
               tab_color: str, font_color: str = '#000000'):
    with pd.ExcelWriter(path, engine='xlsxwriter') as writer:
        df.to_excel(writer, index=False, sheet_name=sheet_name)
        wb = writer.book
        ws = writer.sheets[sheet_name]
        ws.set_tab_color(tab_color)
        hdr_fmt = wb.add_format({
            'bold': True, 'bg_color': tab_color,
            'font_color': font_color,
            'border': 1, 'text_wrap': True,
            'font_name': 'Arial', 'font_size': 10,
        })
        for i, col in enumerate(df.columns):
            ws.write(0, i, col, hdr_fmt)
            if df.empty:
                max_w = len(col) + 4
            else:
                col_max = df[col].dropna().astype(str).str.len().max()
                if pd.isna(col_max):
                    col_max = len(col)
                max_w = max(int(col_max), len(col)) + 4
            ws.set_column(i, i, min(int(max_w), 42))
        ws.freeze_panes(1, 0)
    log.info(f'Wrote {len(df):,} rows -> {path}')


t_start = time.time()
write_xlsx(verified_df, VERIFIED_FILE, 'Verified', '#00B050')
write_xlsx(invalid_df, INVALID_FILE, 'Invalid', '#FF0000', font_color='#FFFFFF')
write_xlsx(combined_df, COMBINED_FILE, 'Combined', '#1F4E78', font_color='#FFFFFF')

elapsed = time.time() - t_start
print(f'\nTotal elapsed: {elapsed:.1f}s  ({elapsed/60:.2f} min)')
print('Output files:')
print(f'  VERIFIED: {VERIFIED_FILE}  ({len(verified_df):,} rows)')
print(f'  INVALID : {INVALID_FILE}   ({len(invalid_df):,} rows)')
print(f'  COMBINED: {COMBINED_FILE}  ({len(combined_df):,} rows)')

13:47:23  INFO      Wrote 393 rows -> ..\..\data\output\verified\verified_addresses_parts_051_051.xlsx
13:47:23  INFO      Wrote 607 rows -> ..\..\data\output\invalid\invalid_addresses_parts_051_051.xlsx
13:47:24  INFO      Wrote 1,000 rows -> ..\..\data\output\combined_addresses_parts_051_051.xlsx



Total elapsed: 1.0s  (0.02 min)
Output files:
  VERIFIED: ..\..\data\output\verified\verified_addresses_parts_051_051.xlsx  (393 rows)
  INVALID : ..\..\data\output\invalid\invalid_addresses_parts_051_051.xlsx   (607 rows)
  COMBINED: ..\..\data\output\combined_addresses_parts_051_051.xlsx  (1,000 rows)


In [70]:
# Quick schema skim for dim_location and alias tables
print('dim_location columns:')
print(dim_location.columns.tolist())
print('\nalias columns:')
print(alias.columns.tolist())

print('\nSample dim_location rows:')
display(dim_location.head(3))

print('\nSample alias rows:')
display(alias.head(5))

dim_location columns:
['psgc_10_digit', 'region_name', 'province_name', 'city_name', 'barangay_name', 'region_code', 'province_code', 'city_code', 'barangay_code']

alias columns:
['pattern', 'replacement']

Sample dim_location rows:


,psgc_10_digit,region_name,province_name,city_name,barangay_name,region_code,province_code,city_code,barangay_code
0,1400101001,Cordillera Administrative Region (CAR),Abra,Bangued,Agtangao,14,1,1,1
1,1400101002,Cordillera Administrative Region (CAR),Abra,Bangued,Angad,14,1,1,2
2,1400101003,Cordillera Administrative Region (CAR),Abra,Bangued,Bañacao,14,1,1,3



Sample alias rows:


,pattern,replacement
0,BRGY,BARANGAY
1,BRGY.,BARANGAY
2,BRG,BARANGAY
3,BRG.,BARANGAY
4,BARRIO,BARANGAY
